In [1]:
!pip install -q kagglehub xgboost networkx plotly

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

!pip install -q torch-geometric

PyTorch version: 2.10.0+cpu
CUDA available: False
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go
import os, glob, warnings, time
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    average_precision_score, roc_auc_score, f1_score, precision_score,
    recall_score, roc_curve, auc
)

import torch
import torch.nn.functional as F

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'All imports loaded! Device: {DEVICE}')

All imports loaded! Device: cpu


In [ ]:
import kagglehub

print("Downloading IBM AML dataset...")
ibm_path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
print(f"  Path: {ibm_path}")
for f in sorted(os.listdir(ibm_path)):
    sz = os.path.getsize(os.path.join(ibm_path, f)) / (1024*1024)
    print(f"    {f} ({sz:.1f} MB)")

print("\nDownloading Czech Financial dataset...")
czech_path = kagglehub.dataset_download("siavashraz/1999-czech-financial-dataset")
print(f"  Path: {czech_path}")
for root, dirs, files in os.walk(czech_path):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / (1024*1024)
        print(f"    {os.path.relpath(fp, czech_path)} ({sz:.1f} MB)")

In [ ]:
ibm_raw = pd.read_csv(os.path.join(ibm_path, 'HI-Small_Trans.csv'))

print("="*65)
print("IBM AML DATASET (HI-Small_Trans.csv)")
print("="*65)
print(f"Shape: {ibm_raw.shape[0]:,} rows × {ibm_raw.shape[1]} columns")
print(f"Memory: {ibm_raw.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"\nColumns & types:")
for col in ibm_raw.columns:
    print(f"  {col:<25} {str(ibm_raw[col].dtype):<10} nunique={ibm_raw[col].nunique():,}")

vc = ibm_raw['Is Laundering'].value_counts()
print(f"\n🎯 Target — Is Laundering:")
print(f"   Clean (0):      {vc[0]:>10,} ({vc[0]/len(ibm_raw)*100:.2f}%)")
print(f"   Laundering (1): {vc[1]:>10,} ({vc[1]/len(ibm_raw)*100:.2f}%)")
print(f"   Imbalance:      1:{vc[0]//vc[1]}")
print(f"\nMissing values: {ibm_raw.isnull().sum().sum()}")
print(f"Duplicate rows:  {ibm_raw.duplicated().sum()}")
ibm_raw.head(3)

In [ ]:
ef find_csv(base, name):
    results = glob.glob(os.path.join(base, '**', name), recursive=True)
    return results[0] if results else None

czech_tables = {}
for fname in ['trans.csv','account.csv','client.csv','disp.csv',
              'district.csv','loan.csv','order.csv','card.csv']:
    fpath = find_csv(czech_path, fname)
    if fpath:
        tname = fname.split('.')[0]
        czech_tables[tname] = pd.read_csv(fpath, sep=';', low_memory=False)
        print(f"{fname:<15} → {czech_tables[tname].shape[0]:>8,} rows × {czech_tables[tname].shape[1]} cols")

cz_trans_raw = czech_tables['trans']
cz_loan = czech_tables['loan']
cz_account = czech_tables['account']

print(f"\n" + "="*65)
print("CZECH TRANSACTIONS (trans.csv)")
print("="*65)
print(f"Shape: {cz_trans_raw.shape[0]:,} rows × {cz_trans_raw.shape[1]} columns")
for col in cz_trans_raw.columns:
    print(f"  {col:<15} {str(cz_trans_raw[col].dtype):<10} nunique={cz_trans_raw[col].nunique():,}")

print(f"\nCzech Loan Status (risk proxy):")
for status, count in cz_loan['status'].value_counts().items():
    desc = {'A':'finished OK','B':'finished NOT paid','C':'running OK','D':'running IN DEBT'}[status]
    print(f"   {status}: {count:>4} — {desc}")

cz_trans_raw.head(3)